<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

<h1 align=center><font size = 5>Assignment: SQL Notebook for Peer Assignment</font></h1>

Estimated time needed: **60** minutes.

## Introduction
Using this Python notebook you will:

1.  Understand the Spacex DataSet
2.  Load the dataset  into the corresponding table in a Db2 database
3.  Execute SQL queries to answer assignment questions 


## Overview of the DataSet

SpaceX has gained worldwide attention for a series of historic milestones. 

It is the only private company ever to return a spacecraft from low-earth orbit, which it first accomplished in December 2010.
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars wheras other providers cost upward of 165 million dollars each, much of the savings is because Space X can reuse the first stage. 


Therefore if we can determine if the first stage will land, we can determine the cost of a launch. 

This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

This dataset includes a record for each payload carried during a SpaceX mission into outer space.


### Download the datasets

This assignment requires you to load the spacex dataset.

In many cases the dataset to be analyzed is available as a .CSV (comma separated values) file, perhaps on the internet. Click on the link below to download and save the dataset (.CSV file):

 <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv" target="_blank">Spacex DataSet</a>



In [1]:
# This notebook uses Python's built-in sqlite3 module.
# No SQLAlchemy installation is required.


### Connect to the database

Let us first load the SQL extension and establish a connection with the database


In [2]:
# No IPython SQL extension is required.
# Queries are executed with sqlite3 and displayed with pandas.


In [3]:
from io import StringIO
from pathlib import Path
import re
import sqlite3

import pandas as pd
import requests

pd.set_option("display.max_columns", None)


In [4]:
# Create a fresh SQLite database connection.
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

print("Connected to SQLite database: my_data1.db")


Connected to SQLite database: my_data1.db


In [5]:
# pandas, requests, and sqlite3 are already imported.


In [6]:
def run_query(query, params=None):
    """Execute a SELECT query and return the result as a DataFrame."""
    return pd.read_sql_query(query, con, params=params)


def normalise_column_name(name):
    """Convert a column heading to a stable SQL-friendly name."""
    text = str(name).strip()
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    return text.strip("_")


def first_existing_column(columns, candidates):
    """Return the first matching column name, ignoring case."""
    lookup = {str(column).lower(): column for column in columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


In [7]:
official_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
)

official_local_files = [
    Path("Spacex.csv"),
    Path("spacex.csv"),
]

web_scraped_local_files = [
    Path("spacex_web_scraped.csv"),
]

df = None
source_description = None

# 1. Prefer an official local CSV when available.
for file_path in official_local_files:
    if file_path.exists():
        df = pd.read_csv(file_path)
        source_description = f"local official dataset: {file_path.resolve()}"
        break

# 2. Otherwise download the official IBM dataset.
if df is None:
    try:
        response = requests.get(official_url, timeout=60)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
        source_description = "official IBM-hosted SpaceX dataset"
    except requests.RequestException:
        df = None

# 3. If the IBM server is unavailable, use the output from Notebook 2.
if df is None:
    for file_path in web_scraped_local_files:
        if file_path.exists():
            scraped = pd.read_csv(file_path)

            # Convert the web-scraping output into the SQL lab schema.
            df = pd.DataFrame({
                "Date": scraped.get("Date"),
                "Time_UTC": scraped.get("Time"),
                "Booster_Version": scraped.get("Version Booster"),
                "Launch_Site": scraped.get("Launch site"),
                "Payload": scraped.get("Payload"),
                "Payload_Mass_KG": scraped.get("Payload mass"),
                "Orbit": scraped.get("Orbit"),
                "Customer": scraped.get("Customer"),
                "Mission_Outcome": scraped.get("Launch outcome"),
                "Landing_Outcome": scraped.get("Booster landing"),
            })

            source_description = (
                f"local web-scraping output: {file_path.resolve()}"
            )
            break

if df is None:
    raise FileNotFoundError(
        "The SpaceX SQL dataset could not be loaded. Place either "
        "'Spacex.csv' or the previously generated "
        "'spacex_web_scraped.csv' in the same folder as this notebook."
    )

# Normalise all incoming column names.
df.columns = [normalise_column_name(column) for column in df.columns]

# Map alternate input headings to the standard schema used by the SQL tasks.
column_candidates = {
    "Date": ["Date"],
    "Time_UTC": ["Time_UTC", "Time"],
    "Booster_Version": [
        "Booster_Version",
        "Version_Booster",
        "BoosterVersion",
    ],
    "Launch_Site": ["Launch_Site", "LaunchSite"],
    "Payload": ["Payload"],
    "Payload_Mass_KG": [
        "PAYLOAD_MASS__KG_",
        "Payload_Mass_KG",
        "Payload_mass",
        "PayloadMass",
    ],
    "Orbit": ["Orbit"],
    "Customer": ["Customer"],
    "Mission_Outcome": [
        "Mission_Outcome",
        "Launch_outcome",
        "Launch_Outcome",
        "Outcome",
    ],
    "Landing_Outcome": [
        "Landing_Outcome",
        "Booster_landing",
        "Booster_Landing",
    ],
}

standardised = pd.DataFrame(index=df.index)

for standard_name, candidates in column_candidates.items():
    source_column = first_existing_column(df.columns, candidates)

    if source_column is None:
        standardised[standard_name] = pd.NA
    else:
        standardised[standard_name] = df[source_column]

df = standardised.copy()

# Clean payload mass values such as "15,600 kg (34,400 lb)".
payload_text = (
    df["Payload_Mass_KG"]
    .astype("string")
    .str.replace(",", "", regex=False)
)

df["Payload_Mass_KG"] = pd.to_numeric(
    payload_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce",
)

# Standardise dates as ISO YYYY-MM-DD where possible.
parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
df.loc[parsed_dates.notna(), "Date"] = (
    parsed_dates[parsed_dates.notna()].dt.strftime("%Y-%m-%d")
)

# Remove completely blank records.
df = df.dropna(how="all").reset_index(drop=True)

required_for_tasks = [
    "Date",
    "Booster_Version",
    "Launch_Site",
    "Payload_Mass_KG",
    "Customer",
    "Mission_Outcome",
    "Landing_Outcome",
]

missing_required = [
    column
    for column in required_for_tasks
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required SQL columns: {missing_required}"
    )

if df.empty:
    raise ValueError("The loaded SpaceX dataset is empty.")

df.to_sql("SPACEXTBL", con, if_exists="replace", index=False)

print("Loaded:", source_description)
print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns")
df.head()


Loaded: official IBM-hosted SpaceX dataset
Dataset shape: 101 rows and 10 columns


,Date,Time_UTC,Booster_Version,Launch_Site,Payload,Payload_Mass_KG,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


**Note:This below code is added to remove blank rows from table**


In [8]:
# Drop any previous cleaned table.
cur.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.commit()

print("Previous SPACEXTABLE removed, when present.")


Previous SPACEXTABLE removed, when present.


In [9]:
# Create the cleaned SQL table and remove blank-date records.
cur.execute("""
    CREATE TABLE SPACEXTABLE AS
    SELECT *
    FROM SPACEXTBL
    WHERE Date IS NOT NULL
      AND TRIM(CAST(Date AS TEXT)) <> ''
""")
con.commit()

row_count = run_query(
    "SELECT COUNT(*) AS Number_of_Rows FROM SPACEXTABLE"
)

print("SPACEXTABLE created successfully.")
row_count


SPACEXTABLE created successfully.


,Number_of_Rows
0,101


## Tasks

Now write and execute SQL queries to solve the assignment tasks.

**Note: If the column names are in mixed case enclose it in double quotes
   For Example "Landing_Outcome"**

### Task 1




##### Display the names of the unique launch sites  in the space mission


In [10]:
# TASK 1: Display the unique launch-site names.
task_1 = run_query("""
    SELECT DISTINCT Launch_Site
    FROM SPACEXTABLE
    WHERE Launch_Site IS NOT NULL
    ORDER BY Launch_Site
""")

task_1


,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E



### Task 2


#####  Display 5 records where launch sites begin with the string 'CCA' 


In [11]:
# TASK 2: Display five records whose launch site begins with "CCA".
task_2 = run_query("""
    SELECT *
    FROM SPACEXTABLE
    WHERE Launch_Site LIKE 'CCA%'
    LIMIT 5
""")

task_2


,Date,Time_UTC,Booster_Version,Launch_Site,Payload,Payload_Mass_KG,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3




##### Display the total payload mass carried by boosters launched by NASA (CRS)


In [12]:
# TASK 3: Total payload mass carried for NASA (CRS).
task_3 = run_query("""
    SELECT
        ROUND(SUM(Payload_Mass_KG), 2) AS Total_Payload_Mass_KG
    FROM SPACEXTABLE
    WHERE Customer = 'NASA (CRS)'
""")

task_3


,Total_Payload_Mass_KG
0,45596.0


### Task 4




##### Display average payload mass carried by booster version F9 v1.1


In [13]:
# TASK 4: Average payload mass carried by booster version F9 v1.1.
task_4 = run_query("""
    SELECT
        ROUND(AVG(Payload_Mass_KG), 2) AS Average_Payload_Mass_KG
    FROM SPACEXTABLE
    WHERE Booster_Version = 'F9 v1.1'
""")

task_4


,Average_Payload_Mass_KG
0,2928.4


### Task 5

##### List the date when the first succesful landing outcome in ground pad was acheived.


_Hint:Use min function_ 


In [14]:
# TASK 5: Date of the first successful ground-pad landing.
task_5 = run_query("""
    SELECT
        MIN(Date) AS First_Successful_Ground_Pad_Landing
    FROM SPACEXTABLE
    WHERE Landing_Outcome = 'Success (ground pad)'
""")

task_5


,First_Successful_Ground_Pad_Landing
0,2015-12-22


### Task 6

##### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000


In [15]:
# TASK 6: Boosters with successful drone-ship landings and
# payload mass greater than 4,000 kg but less than 6,000 kg.
task_6 = run_query("""
    SELECT DISTINCT Booster_Version
    FROM SPACEXTABLE
    WHERE Landing_Outcome = 'Success (drone ship)'
      AND Payload_Mass_KG > 4000
      AND Payload_Mass_KG < 6000
    ORDER BY Booster_Version
""")

task_6


,Booster_Version
0,F9 FT B1021.2
1,F9 FT B1031.2
2,F9 FT B1022
3,F9 FT B1026


### Task 7




##### List the total number of successful and failure mission outcomes


In [16]:
# TASK 7: Count successful and failed mission outcomes.
task_7 = run_query("""
    SELECT
        CASE
            WHEN LOWER(Mission_Outcome) LIKE 'success%'
                THEN 'Success'
            WHEN LOWER(Mission_Outcome) LIKE 'failure%'
                THEN 'Failure'
            ELSE Mission_Outcome
        END AS Mission_Result,
        COUNT(*) AS Total
    FROM SPACEXTABLE
    WHERE Mission_Outcome IS NOT NULL
    GROUP BY Mission_Result
    ORDER BY Total DESC
""")

task_7


,Mission_Result,Total
0,Success,100
1,Failure,1


### Task 8



##### List all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function.


In [17]:
# TASK 8: Booster versions that carried the maximum payload mass.
task_8 = run_query("""
    SELECT DISTINCT Booster_Version, Payload_Mass_KG
    FROM SPACEXTABLE
    WHERE Payload_Mass_KG = (
        SELECT MAX(Payload_Mass_KG)
        FROM SPACEXTABLE
    )
    ORDER BY Booster_Version
""")

task_8


,Booster_Version,Payload_Mass_KG
0,F9 B5 B1048.4,15600
1,F9 B5 B1048.5,15600
2,F9 B5 B1049.4,15600
3,F9 B5 B1049.5,15600
4,F9 B5 B1049.7,15600
5,F9 B5 B1051.3,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1051.6,15600
8,F9 B5 B1056.4,15600
9,F9 B5 B1058.3,15600


### Task 9


##### List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

**Note: SQLLite does not support monthnames. So you need to use  substr(Date, 6,2) as month to get the months and substr(Date,0,5)='2015' for year.**


In [18]:
# TASK 9: Month, failed drone-ship landing outcome, booster version,
# and launch site for records in 2015.
task_9 = run_query("""
    SELECT
        CASE SUBSTR(Date, 6, 2)
            WHEN '01' THEN 'January'
            WHEN '02' THEN 'February'
            WHEN '03' THEN 'March'
            WHEN '04' THEN 'April'
            WHEN '05' THEN 'May'
            WHEN '06' THEN 'June'
            WHEN '07' THEN 'July'
            WHEN '08' THEN 'August'
            WHEN '09' THEN 'September'
            WHEN '10' THEN 'October'
            WHEN '11' THEN 'November'
            WHEN '12' THEN 'December'
        END AS Month,
        Landing_Outcome,
        Booster_Version,
        Launch_Site
    FROM SPACEXTABLE
    WHERE SUBSTR(Date, 1, 4) = '2015'
      AND Landing_Outcome = 'Failure (drone ship)'
    ORDER BY Date
""")

task_9


,Month,Landing_Outcome,Booster_Version,Launch_Site
0,January,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10




##### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.


In [19]:
# TASK 10: Rank landing-outcome counts between the two specified dates.
task_10 = run_query("""
    SELECT
        Landing_Outcome,
        COUNT(*) AS Outcome_Count
    FROM SPACEXTABLE
    WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
      AND Landing_Outcome IS NOT NULL
    GROUP BY Landing_Outcome
    ORDER BY Outcome_Count DESC, Landing_Outcome
""")

task_10


,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Failure (drone ship),5
2,Success (drone ship),5
3,Controlled (ocean),3
4,Success (ground pad),3
5,Failure (parachute),2
6,Uncontrolled (ocean),2
7,Precluded (drone ship),1


In [20]:
# Final validation: confirm that all ten task results were created.
task_results = [
    task_1, task_2, task_3, task_4, task_5,
    task_6, task_7, task_8, task_9, task_10
]

for number, result in enumerate(task_results, start=1):
    if not isinstance(result, pd.DataFrame):
        raise TypeError(f"Task {number} did not return a DataFrame.")

print("All ten SQL tasks executed successfully.")
print("Database table rows:", len(df))


All ten SQL tasks executed successfully.
Database table rows: 101


### Reference Links

* <a href ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20String%20Patterns%20-%20Sorting%20-%20Grouping/instructional-labs.md.html?origin=www.coursera.org">Hands-on Lab : String Patterns, Sorting and Grouping</a>  

*  <a  href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20Built-in%20functions%20/Hands-on_Lab__Built-in_Functions.md.html?origin=www.coursera.org">Hands-on Lab: Built-in functions</a>

*  <a  href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20Sub-queries%20and%20Nested%20SELECTs%20/instructional-labs.md.html?origin=www.coursera.org">Hands-on Lab : Sub-queries and Nested SELECT Statements</a>

*   <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Module%205/DB0201EN-Week3-1-3-SQLmagic.ipynb">Hands-on Tutorial: Accessing Databases with SQL magic</a>

*  <a href= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Module%205/DB0201EN-Week3-1-4-Analyzing.ipynb">Hands-on Lab: Analyzing a real World Data Set</a>




## Author(s)

<h4> Lakshmi Holla </h4>


## Other Contributors

<h4> Rav Ahuja </h4>


<!--
## Change log
| Date | Version | Changed by | Change Description |
|------|--------|--------|---------|
| 2024-07-10 | 1.1 |Anita Verma | Changed Version|
| 2021-07-09 | 0.2 |Lakshmi Holla | Changes made in magic sql|
| 2021-05-20 | 0.1 |Lakshmi Holla | Created Initial Version |
-->


## <h3 align="center"> © IBM Corporation 2021. All rights reserved. <h3/>
